In [ ]:
# 고전 컴퓨터비전 실습 — ③ 분할(Segmentation) 사례2: 자율주행
# [Instance(인스턴스) 방식] 개별 차선을 서로 다른 객체로 구분해 분할
#
# 인스턴스 분할은 시맨틱 분할(실습2)에서 한 걸음 더 나아가, "차선 도색"이라는
# 같은 클래스 안에서도 "왼쪽 차선", "오른쪽 차선", "중앙 점선 차선"을 서로
# 다른 객체(instance)로 구분한다. 특히 중앙 점선은 여러 조각으로 끊어져
# 있으므로, 단순 Connected Components를 그대로 쓰면 점선 조각 하나하나가
# 별개의 "차선"으로 잘못 나뉜다. 이를 해결하기 위해 형태학적 팽창(dilation)
# 으로 끊어진 조각을 먼저 이어붙인 뒤 라벨링하는 어댑터를 사용한다.
#
# Clean Architecture 레이어:
#   [도메인 계층]      — 값 객체 + 추상 인터페이스. cv2 비의존.
#   [유스케이스 계층]   — InstanceSegmentationUseCase. 시맨틱 분할기 + 인스턴스
#                        분리기를 주입받아 "클래스 추출 → 개체 분리"를 조립.
#   [인프라 계층]      — HSV(시맨틱) + 형태학적 병합 라벨링(인스턴스 분리) 구현체.
#   [구성 루트]        — Colab 셀: 합성 도로 영상 생성, DI 조립, 시각화/저장.

## 셀 1: 환경 준비

In [ ]:
# !pip install opencv-python-headless matplotlib numpy --quiet
from __future__ import annotations

import numpy as np
import matplotlib
from matplotlib import font_manager as _font_manager
import glob as _glob
# 한글 폰트가 없는 환경(Colab 등)이면: !apt-get -qq install -y fonts-nanum && fc-cache -f
_candidates = _glob.glob("/usr/share/fonts/truetype/nanum/NanumGothic.ttf") + \
    _glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc") + \
    _glob.glob("/usr/share/fonts/**/NotoSansCJK*.ttc", recursive=True)
_korean_font_name = "sans-serif"
for _font_path in _candidates:
    try:
        _font_manager.fontManager.addfont(_font_path)
        _korean_font_name = _font_manager.FontProperties(fname=_font_path).get_name()
        break
    except Exception:
        continue
matplotlib.rcParams["font.family"] = _korean_font_name
matplotlib.rcParams["axes.unicode_minus"] = False

# ===== [도메인 계층 / Domain Layer] =====

from abc import ABC, abstractmethod
from dataclasses import dataclass


@dataclass(frozen=True)
class SemanticSegmentationResult:
    """실습2와 동일한 개념의 시맨틱 분할 결과 값 객체(이 파일은 독립 실행
    가능해야 하므로 재정의한다)."""
    original_rgb: np.ndarray
    class_mask: np.ndarray
    class_names: dict[int, str]

    def binary_mask_of(self, class_id: int) -> np.ndarray:
        return self.class_mask == class_id


@dataclass(frozen=True)
class InstanceSegmentationResult:
    original_rgb: np.ndarray
    instance_map: np.ndarray
    instance_count: int
    semantic_class_name: str


class SemanticSegmenter(ABC):
    """"영상을 받아 클래스별 라벨 맵을 돌려준다"는 계약 — 실습2의 HSV
    분할기를 그대로 재사용(1단계: 관심 클래스 픽셀 찾기)."""

    @abstractmethod
    def segment(self, image: np.ndarray) -> SemanticSegmentationResult:
        ...


class InstanceSplitter(ABC):
    """"이진 마스크를 받아 개체별 ID로 나눠 돌려준다"는 계약. 실습3의
    WatershedInstanceSplitter와 똑같은 인터페이스를 구현하지만, 내부
    전략은 전혀 다르다(병합 후 라벨링) — LSP를 지키는 서로 다른 구현의
    좋은 예시다."""

    @abstractmethod
    def split(self, binary_mask: np.ndarray) -> tuple[np.ndarray, int]:
        """반환값: (instance_map, instance_count)"""
        ...


# ===== [유스케이스 계층 / Use Case Layer] =====

class InstanceSegmentationUseCase:
    """실습3과 동일한 구조의 유스케이스. "시맨틱 추출 → 인스턴스 분리"
    2단계를 조립하며, 두 단계 모두 추상 인터페이스에만 의존한다(DIP)."""

    def __init__(
        self,
        semantic_segmenter: SemanticSegmenter,
        instance_splitter: InstanceSplitter,
        target_class_id: int,
    ):
        self._semantic_segmenter = semantic_segmenter
        self._instance_splitter = instance_splitter
        self._target_class_id = target_class_id

    def run(self, image: np.ndarray) -> InstanceSegmentationResult:
        semantic_result = self._semantic_segmenter.segment(image)
        target_mask = semantic_result.binary_mask_of(self._target_class_id)
        instance_map, count = self._instance_splitter.split(target_mask)

        return InstanceSegmentationResult(
            original_rgb=semantic_result.original_rgb,
            instance_map=instance_map,
            instance_count=count,
            semantic_class_name=semantic_result.class_names[self._target_class_id],
        )


# ===== [인프라/어댑터 계층 / Infrastructure Layer] =====

import cv2


class HSVColorRoadSegmenter(SemanticSegmenter):
    """실습2와 동일한 HSV 색상 규칙 기반 시맨틱 분할 어댑터."""

    def __init__(
        self,
        lane_value_threshold: int = 190,
        lane_saturation_max: int = 60,
        road_saturation_max: int = 60,
        road_value_range: tuple[int, int] = (50, 190),
    ):
        self._lane_value_threshold = lane_value_threshold
        self._lane_saturation_max = lane_saturation_max
        self._road_saturation_max = road_saturation_max
        self._road_value_range = road_value_range

    def segment(self, image: np.ndarray) -> SemanticSegmentationResult:
        hsv = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
        h, s, v = hsv[..., 0], hsv[..., 1], hsv[..., 2]

        class_mask = np.zeros(image.shape[:2], dtype=np.int32)
        is_road = (s <= self._road_saturation_max) & \
                  (v >= self._road_value_range[0]) & (v <= self._road_value_range[1])
        class_mask[is_road] = 1
        is_lane = (s <= self._lane_saturation_max) & (v >= self._lane_value_threshold)
        class_mask[is_lane] = 2

        return SemanticSegmentationResult(
            original_rgb=image, class_mask=class_mask,
            class_names={0: "하늘/배경", 1: "도로", 2: "차선 도색"},
        )


class MorphologicalMergeInstanceSplitter(InstanceSplitter):
    """세로 방향으로 긴 구조화 요소(structuring element)로 팽창(dilate)시켜
    점선처럼 끊어진 조각들을 먼저 이어붙인 뒤 Connected Components로
    라벨링하는 어댑터. 팽창은 "인스턴스를 나누기 위한 중간 계산"에만
    쓰고, 최종 라벨은 원래 마스크 영역으로 다시 제한해(교집합) 차선
    두께가 부풀려지지 않게 한다."""

    def __init__(self, merge_kernel_height: int = 35, merge_kernel_width: int = 5):
        self._kernel = cv2.getStructuringElement(
            cv2.MORPH_RECT, (merge_kernel_width, merge_kernel_height)
        )

    def split(self, binary_mask: np.ndarray) -> tuple[np.ndarray, int]:
        mask_u8 = binary_mask.astype(np.uint8)
        if mask_u8.sum() == 0:
            return np.zeros_like(mask_u8, dtype=np.int32), 0

        merged = cv2.dilate(mask_u8, self._kernel)
        n_labels, merged_labels = cv2.connectedComponents(merged)

        # 팽창된 라벨을 "원래 마스크 영역"에만 다시 대입 — 두께 왜곡 방지
        instance_map = np.where(mask_u8 > 0, merged_labels, 0).astype(np.int32)
        instance_count = n_labels - 1
        return instance_map, instance_count


class InstanceSegmentationVisualizer:
    """시각화 전담 클래스(SRP). 실습3과 동일한 역할을 수행하되, RGB
    원본을 그대로 쓰는 자율주행 케이스에 맞게 구현한다."""

    def __init__(self, title: str):
        self._title = title
        self._palette = [
            (255, 99, 71), (65, 180, 255), (154, 255, 123),
            (255, 215, 0), (218, 112, 214), (255, 165, 0),
        ]

    def save_result(self, result: InstanceSegmentationResult, out_path: str) -> None:
        import matplotlib.pyplot as plt
        import matplotlib.patches as mpatches

        h, w = result.instance_map.shape
        colored = np.zeros((h, w, 3), dtype=np.uint8)
        for instance_id in range(1, result.instance_count + 1):
            color = self._palette[(instance_id - 1) % len(self._palette)]
            colored[result.instance_map == instance_id] = color

        overlay = result.original_rgb.copy()
        mask_any = result.instance_map > 0
        overlay[mask_any] = colored[mask_any]  # 차선은 얇아서 덧칠이 더 잘 보이도록 100%로 표시

        fig, axes = plt.subplots(1, 2, figsize=(13, 6))
        fig.suptitle(self._title, fontsize=14, fontweight="bold")

        axes[0].imshow(result.original_rgb)
        axes[0].set_title("1. 원본 (합성 도로 장면)")
        axes[0].axis("off")

        axes[1].imshow(overlay)
        axes[1].set_title(f"2. Instance 분할 결과 ({result.semantic_class_name} {result.instance_count}개 개별 구분)")
        axes[1].axis("off")
        legend_patches = [
            mpatches.Patch(color=np.array(self._palette[i % len(self._palette)]) / 255,
                            label=f"차선 #{i + 1}")
            for i in range(result.instance_count)
        ]
        axes[1].legend(handles=legend_patches, loc="upper right", fontsize=9, framealpha=0.9)

        plt.tight_layout()
        plt.savefig(out_path, dpi=120, bbox_inches="tight")
        plt.show()
        print(f"저장 완료: {out_path} / 구분된 개체 수: {result.instance_count}")


# ===== [구성 루트 / Composition Root — Colab 실행 코드] =====

## 셀 2: 합성 도로 장면 생성 (실습2와 동일한 생성 로직 — 좌/우 실선

In [ ]:
# + 중앙 점선 7조각 포함)
def make_synthetic_road_scene(width: int = 400, height: int = 300) -> np.ndarray:
    canvas = np.zeros((height, width, 3), dtype=np.uint8)
    for y in range(height):
        t = y / height
        canvas[y, :] = (int(25 + 10 * t), int(35 + 15 * t), int(70 + 20 * t))

    horizon_y = int(height * 0.45)
    road_pts = np.array([
        [int(width * 0.38), horizon_y], [int(width * 0.62), horizon_y],
        [int(width * 0.95), height], [int(width * 0.05), height],
    ], dtype=np.int32)
    cv2.fillPoly(canvas, [road_pts], color=(92, 92, 96))

    left_line = np.array([
        [int(width * 0.39), horizon_y + 2], [int(width * 0.10), height - 2],
        [int(width * 0.14), height - 2], [int(width * 0.41), horizon_y + 2],
    ], dtype=np.int32)
    right_line = np.array([
        [int(width * 0.61), horizon_y + 2], [int(width * 0.90), height - 2],
        [int(width * 0.86), height - 2], [int(width * 0.59), horizon_y + 2],
    ], dtype=np.int32)
    cv2.fillPoly(canvas, [left_line], color=(245, 245, 245))
    cv2.fillPoly(canvas, [right_line], color=(245, 245, 245))

    n_dashes = 7
    for i in range(n_dashes):
        t0 = i / n_dashes
        t1 = (i + 0.5) / n_dashes
        y0 = int(horizon_y + (height - horizon_y) * t0)
        y1 = int(horizon_y + (height - horizon_y) * t1)
        x_center = width // 2
        half_w = max(2, int(2 + 6 * t0))
        cv2.rectangle(canvas, (x_center - half_w, y0), (x_center + half_w, y1), (245, 245, 245), -1)

    return canvas


road_image = make_synthetic_road_scene(width=400, height=300)
print("합성 도로 영상 크기:", road_image.shape)

## 셀 3: 구체 어댑터 생성 + 의존성 주입(DIP)

In [ ]:
semantic_segmenter: SemanticSegmenter = HSVColorRoadSegmenter(
    lane_value_threshold=190, lane_saturation_max=60,
    road_saturation_max=60, road_value_range=(50, 190),
)
instance_splitter: InstanceSplitter = MorphologicalMergeInstanceSplitter(
    merge_kernel_height=35, merge_kernel_width=5,
)

use_case = InstanceSegmentationUseCase(
    semantic_segmenter=semantic_segmenter,
    instance_splitter=instance_splitter,
    target_class_id=2,  # "차선 도색" 클래스만 개체별로 분리
)

## 셀 4: 실행

In [ ]:
result = use_case.run(road_image)
print(f"구분된 '{result.semantic_class_name}' 개체 수: {result.instance_count}")

## 셀 5: 결과 시각화 및 저장

In [ ]:
visualizer = InstanceSegmentationVisualizer(
    title="[자율주행] Instance 분할 — 개별 차선 구분 (HSV + 형태학적 병합 라벨링)"
)
visualizer.save_result(result, "driving_instance_result.png")

## 셀 6: (참고) Semantic vs Instance 비교 + 병합 전략의 효과 확인

In [ ]:
import scipy.ndimage as ndi  # noqa: E402  (비교용 — 병합 없이 라벨링했다면?)
lane_binary = (result.instance_map > 0)
naive_count, _ = ndi.label(lane_binary)
print("\n=== Semantic vs Instance 비교 ===")
print("Semantic 분할: '차선 도색'이라는 클래스 1개로만 표시 (몇 개 차선인지 알 수 없음)")
print(f"Instance 분할(병합 라벨링 적용): 차선 {result.instance_count}개로 올바르게 구분 "
      f"(좌측/우측 실선 + 중앙 점선 1개)")
print(f"참고: 만약 병합 없이 단순 Connected Components만 썼다면, 점선 조각들이 "
      f"각각 분리되어 훨씬 더 많은(최대 9개) 거짓 '차선'으로 잘못 나뉘었을 것입니다.")